## Notebook 08 — generador con estructura de plantilla real

Objetivo: sintetizar la Memoria Anual siguiendo la estructura fija del
ejemplo del Ayuntamiento (Introducción, Análisis por Plan, Actividad
Operativa, Conclusiones), limitada a 2 páginas.

No se toca redactor_v1.py. El generador hace su propia llamada de
síntesis, usando state["analysis"] (los ConceptoValor ya resueltos,
con trazabilidad) como fuente de datos — no el draft libre del
Redactor. El prompt es genérico: no depende de qué documentos
concretos generaron los datos, solo de la lista concepto->valor.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

### Paso 1: la estructura como contrato verificable

Definimos las 4 secciones como campos de un modelo Pydantic, no solo
como instrucción en texto. Así, si el modelo no devuelve las 4, lo
detectamos por código (falta un campo obligatorio), no a ojo leyendo
el resultado.

In [ ]:
from pydantic import BaseModel, Field

class MemoriaAnual(BaseModel):
    introduccion: str = Field(description="Resumen general del ejercicio, tono institucional, 2-3 frases")
    analisis_planes: str = Field(description="Avances por cada plan estratégico (D.1.1 a D.1.5), con cifras clave integradas en prosa")
    actividad_operativa: str = Field(description="Datos de gestión operativa: empleo, formación, eventos, espacios")
    conclusiones: str = Field(description="Balance general, 3-4 puntos clave, tono positivo pero objetivo")

### Paso 2: el prompt de síntesis

Recibe dos cosas: el ejemplo de estilo (el PDF que nos dio el
Ayuntamiento, usado SOLO como referencia de tono y densidad, nunca
como fuente de datos) y los datos reales del Analista. El límite de
palabras es un proxy para el límite de páginas — hay que calibrarlo
probando, no fiarse a ciegas de que el modelo lo cumpla a la primera.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import os

EJEMPLO_ESTILO = """
Destaca la ampliación de las redes colaborativas, alcanzando las 78
personas integrantes de la Red, muy por encima de la meta prevista de
10. Esta colaboración se ha materializado en la ejecución de 3
proyectos en 2025 (surgidos de 0 en 2024), incluyendo ferias y
jornadas, con un nivel de satisfacción del 85%. La atención a personas
usuarias ha crecido exponencialmente, de 10 en 2024 a 508 en 2025.
"""  # fragmento corto del PDF, solo como referencia de tono/densidad

PROMPT_MEMORIA = """Eres un redactor técnico municipal. A partir de los
datos proporcionados, redacta una Memoria Anual de Actividades siguiendo
ESTRICTAMENTE este estilo (ejemplo de referencia, tono y densidad, no
copies su contenido):

"{ejemplo_estilo}"

Reglas:
- Usa EXCLUSIVAMENTE las cifras que aparecen en los datos proporcionados.
- No inventes cifras. Si un plan no tiene datos suficientes, dilo brevemente.
- Integra las cifras en prosa fluida, como en el ejemplo (nunca listas
  en introducción/análisis; sí puedes usar viñetas cortas SOLO en
  actividad operativa, como en el ejemplo).
- LÍMITE ESTRICTO: el conjunto de las 4 secciones no debe superar
  {presupuesto_palabras} palabras en total — el documento final debe
  caber en 2 páginas.

Datos disponibles:
{datos}
"""

_plantilla_memoria = ChatPromptTemplate.from_messages([("human", PROMPT_MEMORIA)])

### Paso 3: preparar los datos y medir el tamaño

Reutilizamos el Analisis que ya tienes en memoria del kernel de ayer
(un solo documento, el de la Agencia). Es la opción sin coste — antes
de decidir si corremos el pipeline completo con los 3 documentos
reales (que sí gasta cuota), medimos aquí cuánto texto genera solo
con uno, para tener una referencia de escala.

In [ ]:
def _formatear_datos_para_sintesis(analisis) -> str:
    return "\n".join(f"- {d.concepto}: {d.valor}" for d in analisis.datos)

# "resultado" es el que sigue en memoria del kernel desde ayer
datos_texto = _formatear_datos_para_sintesis(resultado["analysis"])
print(f"Nº de ConceptoValor: {len(resultado['analysis'].datos)}")
print(f"Tamaño del texto de datos: {len(datos_texto)} caracteres")

In [ ]:
DATA_DIR_TEST = PROJECT_ROOT / "data" / "raw" / "prueba"
assert DATA_DIR_TEST.exists(), f"No encuentro la carpeta en {DATA_DIR_TEST}"
print("Archivos en la carpeta de prueba:", list(DATA_DIR_TEST.iterdir()))

In [ ]:
from src_agents.graph.workflow import pipeline
from groq import APIStatusError

try:
    resultado = pipeline.invoke({"uploaded_files": [str(DATA_DIR_TEST)]})
    print("Pipeline completo")
    print(f"Conceptos del Analista: {len(resultado['analysis'].datos)}")
except APIStatusError as e:
    print(f"Se cortó: {e}")

### Por qué esto es mejor para lo que necesitamos hoy

agente_ingesta y agente_analista son funciones normales de Python —
no hace falta el grafo de LangGraph para llamarlas, el grafo solo
sirve para encadenar los 5 nodos automáticamente. Aquí las encadenamos
a mano, nosotras, parando justo donde necesitamos (después del
Analista), sin arriesgar el coste de Redactor + Adaptador.

In [ ]:
from src_agents.agents.ingestion import agente_ingesta
from src_agents.agents.analyst import agente_analista

estado_parcial = {"uploaded_files": [str(DATA_DIR_TEST)]}
estado_parcial.update(agente_ingesta(estado_parcial))
estado_parcial.update(agente_analista(estado_parcial))

print(f"Conceptos del Analista: {len(estado_parcial['analysis'].datos)}")

### Se guarda el Analisis para reutilizarlo sin gastar cuota otra vez

In [ ]:
import json

with open(PROJECT_ROOT / "data" / "analisis_prueba.json", "w", encoding="utf-8") as f:
    json.dump(
        [{"concepto": d.concepto, "valor": d.valor, "fuente": d.fuente} for d in estado_parcial["analysis"].datos],
        f, ensure_ascii=False, indent=2,
    )
print("Guardado en data/analisis_prueba.json")

### Se mide el tamaño sin gastar tokens

In [ ]:
def _formatear_datos_para_sintesis(analisis) -> str:
    return "\n".join(f"- {d.concepto}: {d.valor}" for d in analisis.datos)

datos_texto = _formatear_datos_para_sintesis(estado_parcial["analysis"])
print(f"Tamaño del texto de datos: {len(datos_texto)} caracteres")

### Cuánto hay de duplicado

In [ ]:
valores = [d.valor for d in estado_parcial["analysis"].datos]
valores_unicos = set(valores)

print(f"Total de valores: {len(valores)}")
print(f"Valores únicos: {len(valores_unicos)}")
print(f"Duplicados: {len(valores) - len(valores_unicos)}")

### Por qué agrupamos por valor y no por concepto

Un mismo párrafo largo (ej. "Los resultados son positivos...") aparece
bajo varios conceptos distintos ('PRINCIPALES AVANCES (D.1.2...)' en
distintas filas de la tabla). Para la síntesis de la memoria, lo que
importa es no repetir el mismo texto una y otra vez en el prompt —
agrupamos por valor, y de cada grupo nos quedamos con el primer
concepto/fuente como representante (suficiente para trazabilidad; no
hace falta guardar los 5 conceptos que apuntan al mismo párrafo).

In [ ]:
from src_agents.models.state import Analisis, ConceptoValor

vistos = set()
datos_deduplicados = []
for d in estado_parcial["analysis"].datos:
    if d.valor not in vistos:
        vistos.add(d.valor)
        datos_deduplicados.append(d)

analisis_dedup = Analisis(datos=datos_deduplicados, notas=estado_parcial["analysis"].notas)
print(f"De {len(estado_parcial['analysis'].datos)} a {len(analisis_dedup.datos)} conceptos")

datos_texto_dedup = _formatear_datos_para_sintesis(analisis_dedup)
print(f"Tamaño del texto deduplicado: {len(datos_texto_dedup)} caracteres")

### Paso 4: la llamada de síntesis

Primera vez que llamamos al modelo con el prompt de la memoria
estructurada. Usamos salida estructurada (MemoriaAnual) para forzar
las 4 secciones como campos separados, verificables por código.

In [ ]:
modelo_memoria = ChatGroq(
    model=os.environ.get("GROQ_MODEL_MEMORIA", "llama-3.3-70b-versatile"),
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.3,
)
modelo_memoria_estructurado = modelo_memoria.with_structured_output(MemoriaAnual)

prompt_memoria = _plantilla_memoria.invoke({
    "ejemplo_estilo": EJEMPLO_ESTILO,
    "presupuesto_palabras": 600,
    "datos": datos_texto_dedup,
})

memoria_generada = modelo_memoria_estructurado.invoke(prompt_memoria)
print("--- INTRODUCCIÓN ---")
print(memoria_generada.introduccion)
print("\n--- ANÁLISIS POR PLAN ---")
print(memoria_generada.analisis_planes)
print("\n--- ACTIVIDAD OPERATIVA ---")
print(memoria_generada.actividad_operativa)
print("\n--- CONCLUSIONES ---")
print(memoria_generada.conclusiones)

## Reintento

In [ ]:
def _sintetizar_con_reintento(modelo_estructurado, prompt, intentos: int = 3):
    ultimo_error = None
    for intento in range(intentos):
        try:
            return modelo_estructurado.invoke(prompt)
        except Exception as e:
            ultimo_error = e
            print(f"  Intento {intento + 1}/{intentos} falló (tool_use_failed u otro), reintentando...")
    raise ultimo_error

memoria_generada = _sintetizar_con_reintento(modelo_memoria_estructurado, prompt_memoria)

print("--- INTRODUCCIÓN ---")
print(memoria_generada.introduccion)
print("\n--- ANÁLISIS POR PLAN ---")
print(memoria_generada.analisis_planes)
print("\n--- ACTIVIDAD OPERATIVA ---")
print(memoria_generada.actividad_operativa)
print("\n--- CONCLUSIONES ---")
print(memoria_generada.conclusiones)

### Por qué cambiar de "salida estructurada" a "texto con marcadores"

with_structured_output usa function calling por debajo — una capa
extra que puede fallar en el formateo aunque el contenido sea
correcto 

In [ ]:
PROMPT_MEMORIA_TEXTO = """Eres un redactor técnico municipal. A partir de los
datos proporcionados, redacta una Memoria Anual de Actividades siguiendo
ESTRICTAMENTE este estilo (ejemplo de referencia, tono y densidad, no
copies su contenido):

"{ejemplo_estilo}"

Reglas:
- Usa EXCLUSIVAMENTE las cifras que aparecen en los datos proporcionados.
- No inventes cifras. Si un plan no tiene datos suficientes, dilo brevemente.
- Redacta SIEMPRE en párrafos fluidos, sin viñetas ni listas, en ninguna
  sección, integrando las cifras de forma natural en el texto.
- LÍMITE ESTRICTO: el conjunto de las 4 secciones no debe superar
  {presupuesto_palabras} palabras en total.

Responde EXACTAMENTE en este formato, sin nada más:
==INTRODUCCION==
(texto de la introducción)
==ANALISIS_PLANES==
(texto del análisis por plan)
==ACTIVIDAD_OPERATIVA==
(texto de la actividad operativa)
==CONCLUSIONES==
(texto de las conclusiones)

Datos disponibles:
{datos}
"""

_plantilla_memoria_texto = ChatPromptTemplate.from_messages([("human", PROMPT_MEMORIA_TEXTO)])

### parser simple (sin coste, solo python)

In [ ]:
import re

def _parsear_memoria(texto: str) -> dict:
    secciones = {}
    patron = r"==(\w+)==\s*(.*?)(?=\n==\w+==|\Z)"
    for nombre, contenido in re.findall(patron, texto, re.DOTALL):
        secciones[nombre] = contenido.strip()
    return secciones

In [ ]:
modelo_memoria_texto = ChatGroq(
    model=os.environ.get("GROQ_MODEL_MEMORIA", "llama-3.3-70b-versatile"),
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.3,
)

prompt_texto = _plantilla_memoria_texto.invoke({
    "ejemplo_estilo": EJEMPLO_ESTILO,
    "presupuesto_palabras": 600,
    "datos": datos_texto_dedup,
})

respuesta_memoria = modelo_memoria_texto.invoke(prompt_texto)
secciones_memoria = _parsear_memoria(respuesta_memoria.content)

for nombre, texto in secciones_memoria.items():
    print(f"--- {nombre} ---")
    print(texto)
    print()

In [ ]:
total_palabras = sum(len(texto.split()) for texto in secciones_memoria.values())
print(f"Total de palabras: {total_palabras} (presupuesto: 600)")

### Validación

Esta síntesis nueva es un camino distinto al draft del Redactor — nadie
la ha comprobado contra los datos reales todavía. Antes de generar un
documento que podría acabar en manos del Ayuntamiento, conviene saber
si el modelo mantuvo las cifras fieles o se desvió en algún punto,
igual que ya hace reviewer.py con el draft normal. No modificamos su
archivo — copiamos el mismo patrón de comparación, adaptado a esta
síntesis de 4 secciones en vez de a un único draft.

In [ ]:
import unicodedata

def _quitar_acentos(texto: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")

def validar_memoria(secciones: dict, analisis: Analisis) -> list[str]:
    """Comprueba que cada valor del Analista aparezca en ALGUNA de las
    4 secciones generadas. Mismo patrón determinista que reviewer.py,
    aplicado aquí de forma independiente porque esta síntesis es un
    camino distinto al draft del Redactor."""
    texto_completo = " ".join(secciones.values())
    texto_normalizado = _quitar_acentos(texto_completo.lower())

    incidencias = []
    for dato in analisis.datos:
        valor_normalizado = _quitar_acentos(dato.valor.lower())
        if valor_normalizado not in texto_normalizado:
            incidencias.append(f"'{dato.concepto}' = {dato.valor} no aparece en la memoria generada")
    return incidencias

incidencias_memoria = validar_memoria(secciones_memoria, analisis_dedup)
print(f"Incidencias: {len(incidencias_memoria)} de {len(analisis_dedup.datos)} conceptos")
for i in incidencias_memoria[:10]:
    print(f"  - {i}")

In [ ]:
def validar_memoria(secciones: dict, analisis: Analisis, longitud_max: int = 15) -> list[str]:
    """Solo valida valores cortos (candidatos a cifra/dato puntual).
    Los valores largos (nombres de indicador, párrafos narrativos) se
    excluyen — el mismo tipo de falso positivo que ya detectó el equipo
    en reviewer.py con contenido no numérico."""
    texto_completo = " ".join(secciones.values())
    texto_normalizado = _quitar_acentos(texto_completo.lower())

    incidencias = []
    for dato in analisis.datos:
        if len(dato.valor) > longitud_max:
            continue  # probablemente texto narrativo o nombre de indicador, no cifra
        valor_normalizado = _quitar_acentos(dato.valor.lower())
        if valor_normalizado not in texto_normalizado:
            incidencias.append(f"'{dato.concepto}' = {dato.valor} no aparece en la memoria generada")
    return incidencias

incidencias_reales = validar_memoria(secciones_memoria, analisis_dedup)
print(f"Incidencias (solo valores cortos): {len(incidencias_reales)}")
for i in incidencias_reales:
    print(f"  - {i}")

### Paso 5: generar el docx con esta estructura de 4 secciones

Reemplaza el volcado plano del draft por las 4 secciones con formato,
límite de 2 páginas real, e incluye las incidencias como nota final —
mismo criterio de siempre: no ocultar lo que el validador encontró.

In [ ]:
from docx import Document
from docx.shared import Pt

OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

def generar_memoria_estructurada_docx(secciones: dict, incidencias: list, ruta_salida: Path) -> Path:
    doc = Document()
    doc.add_heading("Memoria Anual de Actividades 2025", level=0)
    doc.add_heading("Departamento de Innovación y Empleo — San Sebastián de los Reyes", level=2)

    titulos = {
        "INTRODUCCION": "1. Introducción",
        "ANALISIS_PLANES": "2. Análisis de los Avances por Plan Estratégico",
        "ACTIVIDAD_OPERATIVA": "3. Análisis de la Actividad Operativa",
        "CONCLUSIONES": "4. Desenlace y Conclusiones",
    }
    for clave, titulo in titulos.items():
        doc.add_heading(titulo, level=1)
        doc.add_paragraph(secciones.get(clave, ""))

    if incidencias:
        doc.add_page_break()
        doc.add_heading("Notas de validación (revisión humana)", level=1)
        p = doc.add_paragraph("Datos puntuales del Analista no citados literalmente en el resumen (selección editorial esperable en un resumen ejecutivo, revisar si algún dato clave falta):")
        p.runs[0].italic = True
        for i in incidencias:
            doc.add_paragraph(i, style="List Bullet")

    doc.save(ruta_salida)
    return ruta_salida

ruta_memoria = generar_memoria_estructurada_docx(
    secciones_memoria, incidencias_reales, OUTPUT_DIR / "memoria_estructurada_prueba.docx"
)
print(f"Generado: {ruta_memoria}")